# 9.2 Bagging (Bootstrap Aggregating)

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 9: Ensemble Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-09-ensemble-learning/09.02_bagging.ipynb)

Companion code for Section 9.2 of the book. Run the cells in order from top to bottom.

**Example 9.2.1** · bash

```bash
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, BaggingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import load_breast_cancer, make_regression
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

# ── Bagging from scratch ──────────────────────────────────────────
class BaggingScratch:
    '''Bagging classifier from scratch.'''

    def __init__(self, base_estimator, n_estimators=100, random_state=42):
        self.base_estimator = base_estimator
        self.n_estimators   = n_estimators
        self.rng            = np.random.RandomState(random_state)
        self.estimators_    = []
        self.oob_indices_   = []

    def fit(self, X, y):
        n = len(X)
        self.estimators_ = []
        self.oob_indices_= []

        for _ in range(self.n_estimators):
            # Bootstrap sample
            idx_boot = self.rng.choice(n, size=n, replace=True)
            idx_oob  = np.array(list(set(range(n)) - set(idx_boot)))

            X_boot, y_boot = X[idx_boot], y[idx_boot]

            # Clone and train estimator
            from sklearn.base import clone
            est = clone(self.base_estimator)
            est.fit(X_boot, y_boot)

            self.estimators_.append(est)
            self.oob_indices_.append(idx_oob)

        return self

    def predict(self, X):
        '''Hard majority vote.'''
        predictions = np.array([est.predict(X) for est in self.estimators_])
        # Shape: (n_estimators, n_samples) → majority vote
        from scipy import stats
        return stats.mode(predictions, axis=0, keepdims=True).mode[0]

    def oob_score(self, X, y):
        '''Out-of-bag score - free cross-validation estimate.'''
        n = len(X)
        oob_predictions = np.full((n, len(self.estimators_)), np.nan)

        for t, (est, oob_idx) in enumerate(zip(self.estimators_, self.oob_indices_)):
            if len(oob_idx) > 0:
                oob_predictions[oob_idx, t] = est.predict(X[oob_idx])

        # For each sample, average predictions from estimators that had it as OOB
        oob_final = []
        valid_idx = []
        for i in range(n):
            votes = oob_predictions[i, ~np.isnan(oob_predictions[i])]
            if len(votes) > 0:
                from scipy import stats
                oob_final.append(stats.mode(votes, keepdims=True).mode[0])
                valid_idx.append(i)

        return np.mean(np.array(oob_final) == y[valid_idx])

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


# ── Comparison: Single Tree vs Bagged Trees ───────────────────────
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                       stratify=y, random_state=42)

print('=== Bagging: Single Tree vs Ensemble ===')
print(f'{'Model':<30} {'Train Acc':>10} {'Test Acc':>10} {'OOB Acc':>10}')
print('='*64)

# Single deep tree - overfits
dt_deep = DecisionTreeClassifier(max_depth=None, random_state=42)
dt_deep.fit(X_train, y_train)
print(f'{"Single Tree (full depth)":<30} {dt_deep.score(X_train,y_train):>10.4f} {dt_deep.score(X_test,y_test):>10.4f}
{"N/A":>10}')

# Bagging (from scratch)
bag_scratch = BaggingScratch(DecisionTreeClassifier(random_state=42),
                              n_estimators=100, random_state=42)
bag_scratch.fit(X_train, y_train)
oob = bag_scratch.oob_score(X_train, y_train)
print(f'{"Bagging Scratch (n=100)":<30} {bag_scratch.score(X_train,y_train):>10.4f}
{bag_scratch.score(X_test,y_test):>10.4f} {oob:>10.4f}')

# Sklearn BaggingClassifier
for n_est in [10, 50, 100, 200, 500]:
    bag_sk = BaggingClassifier(DecisionTreeClassifier(),
                                n_estimators=n_est, oob_score=True,
                                random_state=42, n_jobs=-1)
    bag_sk.fit(X_train, y_train)
    print(f'{f"Sklearn Bagging (n={n_est})":<30} {bag_sk.score(X_train,y_train):>10.4f}
{bag_sk.score(X_test,y_test):>10.4f} {bag_sk.oob_score_:>10.4f}')

# ── Effect of base estimator complexity on bagging ────────────────
print('\nBase estimator depth effect on bagging (n=100):')
print(f'{'Max Depth':<12} {'Single Test':>13} {'Bagged Test':>13} {'Improvement':>13}')
print('='*55)
for depth in [1, 2, 3, 5, 7, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    single_test = dt.score(X_test, y_test)

    bag = BaggingClassifier(DecisionTreeClassifier(max_depth=depth),
                             n_estimators=100, oob_score=True,
                             random_state=42, n_jobs=-1)
    bag.fit(X_train, y_train)
    bagged_test = bag.score(X_test, y_test)
    imp = (bagged_test - single_test) * 100
    depth_str = str(depth) if depth else 'None (full)'
    print(f'{depth_str:<12} {single_test:>13.4f} {bagged_test:>13.4f} {imp:>+12.2f}%')

# ── Bias-Variance decomposition via bagging ────────────────────────
print('\nBias-Variance Analysis: Bagging reduces VARIANCE, not bias')
np.random.seed(42)
n_bootstrap = 50
n_test = 200

# Generate many test datasets
X_test_multi = np.random.randn(n_test, 10)
y_test_true  = (X_test_multi[:,0] + X_test_multi[:,1] > 0).astype(int)

for name, clf in [('Single Tree', DecisionTreeClassifier(random_state=42)),
                   ('Bagged Trees', BaggingClassifier(DecisionTreeClassifier(),
                                     n_estimators=50, random_state=42))]:
    predictions = []
    for b in range(n_bootstrap):
        X_tr = np.random.randn(200, 10)
        y_tr = (X_tr[:,0] + X_tr[:,1] > 0).astype(int)
        c = clone(clf)
        c.fit(X_tr, y_tr)
        predictions.append(c.predict(X_test_multi))

    predictions = np.array(predictions)   # (n_bootstrap, n_test)
    # Variance: how much do predictions vary across datasets?
    variance = predictions.var(axis=0).mean()
    # Bias²: how different is mean prediction from truth?
    mean_pred = predictions.mean(axis=0)
    bias_sq   = ((mean_pred - y_test_true)**2).mean()
    print(f'  {name:20s}: Bias²={bias_sq:.4f}  Variance={variance:.4f}  Total={bias_sq+variance:.4f}')
```